# 01 — Kuantum Devre Temelleri
Bu notebook kuantum devremizi görselleştirir ve nasıl çalıştığını adım adım açıklar.
Sunum için kullanılabilir.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pennylane as qml
import numpy as np
import matplotlib.pyplot as plt
from src.quantum_model import load_config, build_quantum_device, make_quantum_circuit

config = load_config()
n_qubits = config['model']['n_qubits']
n_layers = config['model']['n_layers']
print(f'Qubit sayisi: {n_qubits}, Katman sayisi: {n_layers}')

## Kuantum Devreyi Görselleştir

In [ ]:
dev = build_quantum_device(n_qubits)

@qml.qnode(dev)
def demo_circuit(inputs, weights):
    qml.AngleEmbedding(inputs, wires=range(n_qubits))
    qml.StronglyEntanglingLayers(weights, wires=range(n_qubits))
    return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]

weight_shape = qml.StronglyEntanglingLayers.shape(n_layers=n_layers, n_wires=n_qubits)
dummy_inputs  = np.random.uniform(-1, 1, n_qubits)
dummy_weights = np.random.uniform(0, 2*np.pi, weight_shape)

# Devreyi çiz
fig, ax = qml.draw_mpl(demo_circuit)(dummy_inputs, dummy_weights)
ax.set_title('Hibrit QNN — Kuantum Devre')
plt.tight_layout()
plt.savefig('../assets/circuit_diagram.png', dpi=150)
plt.show()
print('Devre assets/circuit_diagram.png olarak kaydedildi')

## Beklenti Değerleri — Farklı Girişler

In [ ]:
results = []
for _ in range(20):
    inp = np.random.uniform(-1, 1, n_qubits)
    out = demo_circuit(inp, dummy_weights)
    results.append(out)

results = np.array(results)
fig, axes = plt.subplots(1, n_qubits, figsize=(12, 3))
for i, ax in enumerate(axes):
    ax.hist(results[:, i], bins=10, color='#7c3aed', edgecolor='white')
    ax.set_title(f'Qubit {i}')
    ax.set_xlabel('PauliZ beklentisi')
plt.suptitle('Rastgele girişler için kuantum çıktı dağılımı')
plt.tight_layout()
plt.show()